In [ ]:
import re
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment

BASE_URL = "https://www.sports-reference.com"


def clean_team_name(name: str) -> str:
    """Normalize team names by removing seeds and annotation markers."""
    if name is None:
        return ""
    cleaned = re.sub(r"\*", "", str(name))
    cleaned = re.sub(r"\(\d+\)", "", cleaned)
    return cleaned.replace("\u00a0", " ").strip()


In [ ]:
def get_team_stats_from_csv(year: int) -> pd.DataFrame:
    """Scrape team stats for a season, using the visible table or a commented CSV fallback."""
    url = f"{BASE_URL}/cbb/seasons/men/{year}-school-stats.html"
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table", id="basic_school_stats")

    if table is not None:
        df = pd.read_html(str(table))[0]
        df.columns = [col[1] if isinstance(col, tuple) else col for col in df.columns]
    else:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        csv_text = next(
            (comment for comment in comments if "csv_basic_school_stats" in comment or "csv_adv_school_stats" in comment),
            None,
        )
        if csv_text is None:
            raise ValueError("Could not find basic_school_stats table or CSV fallback.")

        df = pd.read_csv(StringIO(csv_text))

    df["School"] = (
        df["School"]
        .astype(str)
        .map(clean_team_name)
    )

    df = df[~df["School"].isin(["School", "Overall"])].copy()

    df = df.rename(columns={
        "School": "team",
        "Conf": "conference",
        "W": "wins",
        "L": "losses",
        "W-L%": "win_pct",
        "PS/G": "points_for",
        "PA/G": "points_against",
    })

    if "team" in df.columns:
        df = df[df["team"].str.endswith("NCAA")].copy()
        df["team"] = df["team"].str.replace(r"\s*NCAA$", "", regex=True).str.strip()

    df = df.drop(columns=[col for col in ["Rk", "Rank"] if col in df.columns])
    df["year"] = year

    return df


In [ ]:
def get_tournament_games(year: int) -> pd.DataFrame:
    """Scrape NCAA tournament results for one season and return a game-level DataFrame."""
    url = f"{BASE_URL}/cbb/postseason/{year}-ncaa.html"
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    games = []

    for round_div in soup.find_all("div", class_="round"):
        for game_div in round_div.find_all("div", recursive=False):
            team_divs = [div for div in game_div.find_all("div", recursive=False) if div.find("a")]
            if len(team_divs) < 2:
                continue

            def extract_team_name(div):
                link = div.find("a")
                return clean_team_name(link.text) if link else ""

            def extract_seed(div):
                seed_span = div.find("span")
                if seed_span is None:
                    return None
                seed_text = seed_span.text.strip()
                return int(seed_text) if seed_text.isdigit() else None

            def extract_score(div):
                score_text = " ".join(a.text.strip() for a in div.find_all("a"))
                matches = re.findall(r"\d+", score_text)
                return int(matches[-1]) if matches else None

            if "winner" in (team_divs[0].get("class") or []):
                winner_div, loser_div = team_divs[0], team_divs[1]
            elif "winner" in (team_divs[1].get("class") or []):
                winner_div, loser_div = team_divs[1], team_divs[0]
            else:
                score0 = extract_score(team_divs[0])
                score1 = extract_score(team_divs[1])
                if score0 is None or score1 is None:
                    continue
                if score0 > score1:
                    winner_div, loser_div = team_divs[0], team_divs[1]
                else:
                    winner_div, loser_div = team_divs[1], team_divs[0]

            games.append({
                "season": year,
                "winner": extract_team_name(winner_div),
                "winner_seed": extract_seed(winner_div),
                "loser": extract_team_name(loser_div),
                "loser_seed": extract_seed(loser_div),
            })

    return pd.DataFrame(games)


In [ ]:
# If additional name normalization is needed after scraping, use the helper below.
# These lines are intentionally commented out because team names are already cleaned during extraction.
#
# games["winner"] = games["winner"].apply(clean_team_name)
# games["loser"] = games["loser"].apply(clean_team_name)


In [ ]:
def build_dataset(start_year: int, end_year: int):
    """Build the dataset for a range of seasons and return team and game DataFrames."""
    all_teams = []
    all_games = []

    for year in range(start_year, end_year + 1):
        print(f"Scraping {year}...")
        teams = get_team_stats_from_csv(year)
        games = get_tournament_games(year)

        all_teams.append(teams)
        all_games.append(games)

    teams_df = pd.concat(all_teams, ignore_index=True)
    games_df = pd.concat(all_games, ignore_index=True)

    return teams_df, games_df


In [ ]:
# Run the scraper and save results to CSV files.
teams_df, games_df = build_dataset(2015, 2025)

teams_df.to_csv("teams.csv", index=False)
games_df.to_csv("tournament_games.csv", index=False)
